# ДЗ №3. YOLO11n на трудных негативах Open Images, v2

Первый запуск (`yolo11n_hardneg`) улучшил разделение предмета и похожего предмета
(AUC на трудных негативах Open Images 0,72 → 0,75) и recall, но FAR на похожих предметах
при пороге COCO не снизился. Здесь проверяется, даст ли эффект в 2–3 раза больше данных.

**Критерий успеха (зафиксирован до запуска).** На Open Images test средний AUC
«предмет против трудного негатива» выше, чем у `yolo11n_hardneg` (0,751), а recall
на COCO val2017 при консервативном пороге не ниже 82%.
Данные как в первом запуске (`yolo11n_hardneg`, коммит `41e6245`), с одной правкой: фото
Open Images train выбираются только из подмножества с рамками. Человеческие метки уровня
фото есть примерно для 9 млн снимков, а на S3 лежат только ~1,7 млн фото с рамками; в первом
запуске из-за этого не скачались 56% выбранных фото и в обучение попало ~7,7 тыс. вместо ~20 тыс.

- **трудные негативы** — проверенная человеком отрицательная метка класса: фото попало
  в разметку как кандидат классификатора, то есть на нём похожий предмет;
- **положительные** — проверенная положительная метка с рамками.

Оптимизатор и расписание как в первом запуске: SGD, lr0 = 0,002, разогрев 1 эпоха,
первые 10 слоёв заморожены, 30 эпох.

**Настройки Kaggle:** Accelerator — GPU T4, Internet — On.
**Входные данные:** датасет COCO 2017 с `train2017` и `instances_train2017.json`.

In [ ]:
import subprocess
import sys

# Ветка или коммит репозитория с пакетом photoquest_eval.
REPO_REF = "hw_3"
_ = subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"photoquest-eval[yolo] @ git+https://github.com/plekhovrodion/mlsd@{REPO_REF}",
    ]
)
REPO_COMMIT = subprocess.check_output(
    ["git", "ls-remote", "https://github.com/plekhovrodion/mlsd", REPO_REF], text=True
).split()[0]
print("photoquest_eval:", REPO_REF, REPO_COMMIT)

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import shutil  # noqa: F401 — ячейка общая для обоих ноутбуков
import time
import urllib.request
import zipfile
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from photoquest_eval.config import (  # noqa: F401 — ячейка общая для обоих ноутбуков
    HOME_CLASSES,
    IMAGE_SIZES,
    SEED,
    TARGETS,
    YOLO_SETTINGS,
)

STARTED_UTC = datetime.now(UTC).isoformat()
STARTED = time.perf_counter()
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle Settings → Accelerator → GPU T4")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def package_versions(*names):
    return {name: importlib.metadata.version(name) for name in names}

In [ ]:
from ultralytics import YOLO

from photoquest_eval.data import (
    OPENIMAGES_URLS,
    boxable_image_ids,
    coco_to_yolo,
    download_openimages,
    openimages_to_yolo,
    sample_hard_negatives,
    scan_openimages_boxes,
    scan_openimages_labels,
    write_yolo_yaml,
)

OUTPUT = Path("/kaggle/working/hw3_train_hardneg_v2")
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_NAME = "yolo11n_hardneg_v2"

# Гиперпараметры фиксируются до обучения и сохраняются в metadata.json.
TRAIN_SETTINGS = {
    "base_model": "yolo11n.pt",
    "epochs": 30,
    "patience": 10,
    "imgsz": 640,
    "batch": 32,
    "optimizer": "SGD",
    "lr0": 0.002,
    "lrf": 0.1,
    "warmup_epochs": 1,
    "freeze": 10,
    "seed": SEED,
    "deterministic": True,
    "workers": 4,
    "save_period": 5,
}
DATA_SETTINGS = {
    "negative_fraction": 0.1,
    "val_fraction": 0.1,
    "seed": SEED,
    "max_side": TRAIN_SETTINGS["imgsz"],
}
OI_TRAIN_SAMPLE = {"positives_per_class": 800, "negatives_per_class": 1500, "seed": SEED}

## Данные

### COCO train2017 — как в `02_train_yolo11n.ipynb`

In [ ]:
paths = sorted(Path("/kaggle/input").rglob("instances_train2017.json"))
if not paths:
    raise FileNotFoundError("Добавьте во входные данные датасет COCO 2017 с train2017")
TRAIN_ANNOTATIONS = paths[0]
coco_root = TRAIN_ANNOTATIONS.parent.parent
TRAIN_IMAGES = next(
    p for p in (coco_root / "train2017", coco_root / "images" / "train2017") if p.is_dir()
)
annotation_hash = sha256(TRAIN_ANNOTATIONS)
DATASET = Path("/tmp/hardneg_yolo")
copy_started = time.perf_counter()
coco_stats = coco_to_yolo(TRAIN_ANNOTATIONS, TRAIN_IMAGES, DATASET, TARGETS, **DATA_SETTINGS)
print(f"COCO: {(time.perf_counter() - copy_started) / 60:.1f} мин")
display(pd.DataFrame(coco_stats).T)

### Open Images train

Файлы разметки train весят 2,7 и 2,3 ГБ; из них читаются только строки 10 целевых классов.

In [ ]:
OI_CACHE = Path("/tmp/openimages_train")
OI_CACHE.mkdir(exist_ok=True)
oi_started = time.perf_counter()
csv_paths = {}
for key in ("train_image_labels", "train_boxes"):
    csv_paths[key] = OI_CACHE / Path(OPENIMAGES_URLS[key]).name
    if not csv_paths[key].exists():
        urllib.request.urlretrieve(OPENIMAGES_URLS[key], csv_paths[key])
oi_status = scan_openimages_labels(csv_paths["train_image_labels"], TARGETS)
# На S3 лежат только фото с рамками (~1,7 млн из ~9 млн с метками): выбираем только из них.
boxable = boxable_image_ids(csv_paths["train_boxes"])
labelled_images = len(oi_status)
oi_status = {image_id: s for image_id, s in oi_status.items() if image_id in boxable}
print(f"Фото с метками целевых классов: {labelled_images:,}, из них с рамками: {len(oi_status):,}")
availability = pd.DataFrame(
    {
        "verified_positive": [sum(s.get(t) == 1 for s in oi_status.values()) for t in TARGETS],
        "verified_negative": [sum(s.get(t) == 0 for s in oi_status.values()) for t in TARGETS],
    },
    index=TARGETS,
)
display(availability)
oi_ids = sample_hard_negatives(oi_status, TARGETS, **OI_TRAIN_SAMPLE)
oi_boxes = scan_openimages_boxes(csv_paths["train_boxes"], oi_ids, TARGETS)
failed = download_openimages(
    oi_ids, OI_CACHE / "images", max_side=TRAIN_SETTINGS["imgsz"], split="train"
)
oi_stats = openimages_to_yolo(
    oi_ids,
    oi_status,
    oi_boxes,
    OI_CACHE / "images",
    DATASET,
    TARGETS,
    DATA_SETTINGS["val_fraction"],
    SEED,
)
oi_seconds = time.perf_counter() - oi_started
print(
    f"Open Images: выбрано {len(oi_ids):,}, не скачалось {len(failed)}, {oi_seconds / 60:.1f} мин"
)
display(pd.DataFrame(oi_stats).T)
pd.Series(oi_ids, name="ImageID").to_csv(OUTPUT / "openimages_train_ids.csv", index=False)
DATASET_YAML = write_yolo_yaml(DATASET, TARGETS)

## Обучение

In [ ]:
CHECKPOINTS = {
    "yolov8n.pt": (
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt",
        "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36",
    ),
    "yolo11n.pt": (
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt",
        "0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1",
    ),
    "yolo11s.pt": (
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11s.pt",
        "85a76fe86dd8afe384648546b56a7a78580c7cb7b404fc595f97969322d502d5",
    ),
}


def get_checkpoint(filename: str, folder: Path) -> Path:
    """Скачивает предобученные веса и сверяет SHA-256 с прогоном ДЗ №2."""
    url, expected = CHECKPOINTS[filename]
    path = folder / filename
    folder.mkdir(parents=True, exist_ok=True)
    if not path.is_file():
        urllib.request.urlretrieve(url, path)
    if sha256(path) != expected:
        raise ValueError(f"Не совпал SHA-256 весов {filename}")
    return path

In [ ]:
base = get_checkpoint(TRAIN_SETTINGS["base_model"], OUTPUT / "pretrained")
model = YOLO(str(base))
train_started = time.perf_counter()
results = model.train(
    data=str(DATASET_YAML),
    project=str(OUTPUT),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    **{k: v for k, v in TRAIN_SETTINGS.items() if k != "base_model"},
)
train_seconds = time.perf_counter() - train_started
RUN = OUTPUT / RUN_NAME
BEST = RUN / "weights" / "best.pt"
print(f"Обучение заняло {train_seconds / 3600:.2f} ч, лучшие веса: {BEST}")

## Кривые обучения

In [ ]:
import matplotlib.pyplot as plt

history = pd.read_csv(RUN / "results.csv")
history.columns = history.columns.str.strip()
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for axis, part in zip(axes[:3], ("box", "cls", "dfl"), strict=True):
    axis.plot(history["epoch"], history[f"train/{part}_loss"], label="train")
    axis.plot(history["epoch"], history[f"val/{part}_loss"], label="val")
    axis.set(title=f"{part}_loss", xlabel="эпоха")
    axis.legend()
axes[3].plot(history["epoch"], history["metrics/mAP50(B)"], label="mAP@0.50")
axes[3].plot(history["epoch"], history["metrics/mAP50-95(B)"], label="mAP@0.50:0.95")
axes[3].set(title="Качество на val", xlabel="эпоха")
axes[3].legend()
fig.tight_layout()
fig.savefig(OUTPUT / "training_curves.png", dpi=150)
plt.show()
history.to_csv(OUTPUT / "training_history.csv", index=False)
display(history.tail(3))

## Экспорт в ONNX

In [ ]:
EXPORTS = OUTPUT / "export"
EXPORTS.mkdir(exist_ok=True)
shutil.copy(BEST, EXPORTS / f"{RUN_NAME}.pt")
export_rows = []
for size in IMAGE_SIZES:
    exported = Path(YOLO(str(BEST)).export(format="onnx", imgsz=size, simplify=True))
    target = EXPORTS / f"{RUN_NAME}_{size}.onnx"
    shutil.move(exported, target)
    export_rows.append(
        {
            "file": target.name,
            "imgsz": size,
            "MiB": target.stat().st_size / 2**20,
            "sha256": sha256(target),
        }
    )
export_rows.append(
    {
        "file": f"{RUN_NAME}.pt",
        "imgsz": 640,
        "MiB": (EXPORTS / f"{RUN_NAME}.pt").stat().st_size / 2**20,
        "sha256": sha256(EXPORTS / f"{RUN_NAME}.pt"),
    }
)
exports = pd.DataFrame(export_rows)
exports.to_csv(OUTPUT / "exports.csv", index=False)
display(exports)

In [ ]:
metadata = {
    "repo_ref": REPO_REF,
    "repo_commit": REPO_COMMIT,
    "train_annotations_sha256": annotation_hash,
    "targets": list(TARGETS),
    "data_settings": DATA_SETTINGS,
    "coco_split_stats": coco_stats,
    "openimages_train_sample": OI_TRAIN_SAMPLE,
    "openimages_train_images": len(oi_ids),
    "openimages_boxable_only": True,
    "openimages_labelled_images": labelled_images,
    "openimages_boxable_labelled_images": len(oi_status),
    "openimages_failed_downloads": len(failed),
    "openimages_split_stats": oi_stats,
    "openimages_csv_sha256": {key: sha256(path) for key, path in csv_paths.items()},
    "train_settings": TRAIN_SETTINGS,
    "best_epoch": int(history.loc[history["metrics/mAP50-95(B)"].idxmax(), "epoch"]),
    "epochs_run": int(history["epoch"].max()),
    "openimages_prep_seconds": round(oi_seconds, 1),
    "train_seconds": round(train_seconds, 1),
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "packages": package_versions(
        "torch", "ultralytics", "numpy", "onnx", "onnxruntime", "onnxslim"
    ),
    "started_utc": STARTED_UTC,
    "elapsed_seconds": round(time.perf_counter() - STARTED, 1),
}
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
archive = Path(f"/kaggle/working/{OUTPUT.name}_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT.rglob("*")):
        relative = path.relative_to(OUTPUT)
        if path.is_file() and not {"pretrained", "weights"} & set(relative.parts):
            z.write(path, relative)
print(json.dumps(metadata, indent=2, ensure_ascii=False))